# SDialog Tutorial: Integrating a RAG System via Tool Calling

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/00_overview/8.rag_tool_integration.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

A step-by-step guide to connecting an external knowledge base (RAG) to an SDialog agent using the native tool system.

## Introduction

LLMs generate responses from knowledge encoded during pre-training. This works well for general tasks, but it becomes insufficient when the agent needs to answer questions about specific, up-to-date, or domain-restricted information — such as university course catalogues, internal procedures, or product databases.

**Retrieval-Augmented Generation (RAG)** addresses this by connecting the model to an external document index at inference time. The model first retrieves a small set of relevant text snippets, then uses those snippets as context to produce a grounded response.

This tutorial shows how to integrate a RAG backend with an SDialog agent using SDialog's native tool system. The example domain is a **university counselling chatbot** that answers questions about Italian universities and courses.

What you will learn:
- How SDialog's tool system works and what requirements a tool function must meet
- How to write a retrieval tool that calls an external search endpoint
- Why the function name and docstring are critical for correct LLM behaviour
- How to build and run a `Counsellor` agent that retrieves information autonomously


---

## 1. Setup

### 1.1 Install dependencies

In [ ]:
import os
from IPython import get_ipython

os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")

if "google.colab" in str(get_ipython()):
    print("Running on Colab")

    !apt-get install -y zstd
    !curl -fsSL https://ollama.com/install.sh | sh

    if not os.path.isdir("/content/sdialog"):
        !git clone https://github.com/idiap/sdialog.git /content/sdialog
    else:
        print("sdialog directory already exists, skipping clone.")

    !pip install -e /content/sdialog --config-settings editable_mode=compat -q

    # Fix: assicura che Python usi il sorgente corretto di sdialog
    import sys
    for key in list(sys.modules.keys()):
        if "sdialog" in key:
            del sys.modules[key]
    if "/content/sdialog/src" not in sys.path:
        sys.path.insert(0, "/content/sdialog/src")

else:
    print("Running in Jupyter Notebook")
    get_ipython().system = os.system

### 1.2 Start the Ollama server and download the model

If you plan to use a local Ollama model, start the server in the background. If you prefer to use an OpenAI key instead, skip this cell and adjust the model string in Section 1.3.

In [ ]:
import requests, time, subprocess

# Start the Ollama server in the background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait until the server is ready (up to 30 seconds)
for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama server is ready.")
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time.")

# Pull the model (skipped automatically if already present)
subprocess.run(["ollama", "pull", "llama3.2:3b"], check=True)

# Verify the model is available
r = requests.get("http://localhost:11434/api/tags", timeout=10)
models = [m["name"] for m in r.json().get("models", [])]
print("Available models:", models)

Ollama server is ready.
Available models: ['llama3.2:3b']


### 1.3 Configure the LLM backend

Set the global model used by SDialog. Any backend supported by SDialog can be used here, for example:
- `"ollama:llama3.1:8b"` — a small local model suitable for tool calling
- `"openai:gpt-4.1"` — an OpenAI model (requires `OPENAI_API_KEY` to be set)

> **Note:** The model must support tool calling natively. Models that do not expose a function-calling interface will not invoke tools reliably. LLaMA 3.1 and later, Qwen 2.5 and later, and most OpenAI models all support it.

---

## 2. How SDialog's Tool System Works

SDialog allows plain Python functions to be passed as tools via the `tools` parameter of the `Agent` constructor. Internally, SDialog wraps each function using LangChain's tool decorator, which makes the function available to the LLM as a callable action.

There are two rules that every tool function must follow:

1. **Do not pre-decorate** the function with `@tool`. SDialog applies the decorator itself. Adding it manually causes a conflict.
2. **Always include a docstring.** The LLM has no other way to understand what the function does, when to call it, and how to interpret its output. The docstring is not optional documentation — it is the function's interface to the model.

The **function name** carries equal importance. It is the first thing the LLM reads when deciding which tool to invoke. A name like `search_italian_university_data` is self-explanatory; a name like `search_kb` or `retrieve` is not.

---

## 3. Setting Up the Retrieval Service

The retrieval service is expected at:

```
GET http://<host>/search
    ?index_name=rag_chunks
    &query=<your query here>
    &top_k=<number of results>
```

and returns:

```json
{"documents": ["snippet 1", "snippet 2", "snippet 3"]}
```

Choose one of the two options below depending on your setup. **Only run one of the two sections** (3.1 or 3.2).

### Mock server

The cell below starts a lightweight in-process HTTP server that mimics the retrieval service interface. It serves a small static knowledge base about Italian universities and runs entirely inside the Colab or Jupyter runtime — no external service required.

In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs

KNOWLEDGE_BASE = [
    "The University of Catania offers undergraduate programmes in Computer Science, Engineering, Medicine, and Law. Enrolment opens each year in September.",
    "The University of Bologna (Alma Mater Studiorum) is the oldest university in the world, founded in 1088. It offers over 200 degree programmes across 11 schools.",
    "The Politecnico di Milano ranks among the top technical universities in Europe and is particularly renowned for its Engineering and Architecture programmes.",
    "Italian universities distinguish between a Laurea Triennale (3-year bachelor's degree) and a Laurea Magistrale (2-year master's degree).",
    "The ECTS credit system is used across all Italian universities. A full academic year corresponds to 60 ECTS credits.",
    "Students can apply for the DSU (Diritto allo Studio Universitario) scholarship, which covers tuition fees and provides a monthly allowance based on household income.",
    "Erasmus+ exchanges are available at most Italian universities. Students typically spend one or two semesters at a partner institution abroad.",
    "The University of Rome La Sapienza is the largest university in Italy and one of the largest in Europe, with over 100,000 enrolled students.",
    "The Scuola Normale Superiore in Pisa is a highly selective institution offering advanced programmes in Sciences and Humanities.",
    "Most Italian universities require a high school diploma (diploma di maturita) or an equivalent foreign qualification for undergraduate admission.",
]


class MockRetrievalServer(HTTPServer):
    allow_reuse_address = True


class MockRetrievalHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urlparse(self.path)
        params = parse_qs(parsed.query)

        query = params.get("query", [""])[0].lower()
        top_k = int(params.get("top_k", ["3"])[0])

        scored = [
            (sum(word in doc.lower() for word in query.split()), doc)
            for doc in KNOWLEDGE_BASE
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        top_docs = [doc for _, doc in scored[:top_k]]

        body = json.dumps({"documents": top_docs}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, format, *args):
        pass  # Suppress request logs


mock_server = MockRetrievalServer(("localhost", 7997), MockRetrievalHandler)
threading.Thread(target=mock_server.serve_forever, daemon=True).start()

RETRIEVAL_ENDPOINT = "http://localhost:7997/search"
print(f"Mock retrieval service started at {RETRIEVAL_ENDPOINT}")

### 3.2 Alternative: use a real service via ngrok (optional, advanced)

> **Skip this section if you ran the mockup.**

If you have the retrieval service already running on your local machine and want to connect it to a Colab session, you can expose it over the internet using [ngrok](https://ngrok.com). ngrok creates a public HTTPS tunnel to a port on your local machine.

**Prerequisites:**
- The retrieval service is running locally on port `7997`.
- You have a free ngrok account and your authtoken (available at https://dashboard.ngrok.com/get-started/your-authtoken).

**Step 1 — on your local machine**, open a terminal and run:

```bash
# Install ngrok if not already installed
# macOS:  brew install ngrok
# Linux:  snap install ngrok
# or download from https://ngrok.com/download

ngrok config add-authtoken YOUR_AUTHTOKEN_HERE
ngrok http 7997
```

ngrok will display a forwarding URL such as:

```
Forwarding   https://abc123.ngrok-free.app -> http://localhost:7997
```

**Step 2 — in this notebook**, paste that URL into the cell below and run it:

Replace with the forwarding URL shown by ngrok
NGROK_URL = "https://abc123.ngrok-free.app"

RETRIEVAL_ENDPOINT = f"{NGROK_URL}/search"
print(f"Retrieval endpoint set to: {RETRIEVAL_ENDPOINT}")

### 3.3 Verify the service

Whichever option you chose, run this cell to confirm that the retrieval service responds correctly before proceeding.

In [ ]:
import requests

response = requests.get(
    RETRIEVAL_ENDPOINT,
    params={"index_name": "rag_chunks", "query": "scholarship", "top_k": 2}
)
print(json.dumps(response.json(), indent=2))

{
  "documents": [
    "Students can apply for the DSU (Diritto allo Studio Universitario) scholarship, which covers tuition fees and provides a monthly allowance based on household income.",
    "The University of Catania offers undergraduate programmes in Computer Science, Engineering, Medicine, and Law. Enrolment opens each year in September."
  ]
}


---

## 4. Defining the Retrieval Tool

The tool function wraps the HTTP call to the retrieval service and normalises the response into a flat dictionary that the LLM can easily reference in its reply.

Two design choices are worth noting before reading the code:

- **`docs_k` is typed as `str`**, not `int`. LLMs generate tool arguments as text and may pass numeric values as strings even when the annotation says otherwise. Accepting a string and converting internally with `int(docs_k)` prevents type errors across different model backends. Exposing this parameter at all lets the model adjust retrieval depth to the complexity of the question.
- **Errors return a dict, not an exception.** If the HTTP call fails, the function returns `{"error": "..."}` rather than raising. This keeps the return type consistent and allows the agent to acknowledge the failure in natural language rather than crashing.

In [ ]:
import requests


def search_italian_university_data(query: str, docs_k: str = "3") -> dict:
    """
    Search the university counselling knowledge base for guidelines
    and information relevant to the student's situation.

    Use this tool whenever the conversation requires information about:
    - Universities and Departments in the Italian territory
    - Available resources (offices, contacts, scheduling)

    Args:
        query (str):
            A short, self-contained question or keyword phrase describing
            what information is needed.
        docs_k (str):
            Number of document snippets to retrieve (default: "3").

    Returns:
        dict: A dictionary where each key is "doc1", "doc2", ... "docN" and
              the corresponding value is the text of that retrieved snippet.
              Returns {"error": "<message>"} if the retrieval service fails.
    """
    try:
        top_k = int(docs_k)
    except (ValueError, TypeError):
        top_k = 3

    payload = {
        "index_name": "rag_chunks",
        "query": query,
        "top_k": top_k,
    }

    try:
        response = requests.get(RETRIEVAL_ENDPOINT, params=payload, timeout=10)
        response.raise_for_status()
        results = response.json()
    except requests.RequestException as exc:
        return {"error": str(exc)}

    if isinstance(results, dict) and "documents" in results:
        items = results["documents"]
    else:
        items = results

    snippets = [item if isinstance(item, str) else item.get("text", "") for item in items]
    return {f"doc{i + 1}": text for i, text in enumerate(snippets)}

Verify the tool returns the expected format:

In [ ]:
result = search_italian_university_data("engineering programmes", docs_k="2")
for key, value in result.items():
    print(f"{key}: {value}")

doc1: The University of Catania offers undergraduate programmes in Computer Science, Engineering, Medicine, and Law. Enrolment opens each year in September.
doc2: The Politecnico di Milano ranks among the top technical universities in Europe and is particularly renowned for its Engineering and Architecture programmes.


---

## 5. Building the Counsellor Agent

### 5.1 Define the persona

In [ ]:
from sdialog.agents import Agent, BasePersona


class Counsellor(BasePersona):
    """University Counsellor"""
    name: str = "Counsellor"

    def prompt(self) -> str:
        return (
            "You are a university counsellor helping students choose their courses "
            "and understand the Italian university system. "
            "Always answer in English. "
            "When you need factual information about universities or departments, "
            "use the available search tool before answering."
        )

### 5.2 Instantiate the agent

The tool function is passed via the `tools` parameter. SDialog wraps it internally — no `@tool` decorator is needed.

In [ ]:
agent = Agent(
    persona=Counsellor(),
    model="openai:llama3.2:3b",
    openai_api_base="http://localhost:11434/v1",
    openai_api_key="EMPTY",
    tools=[search_italian_university_data],
)

---

## 6. Running the Agent

### 6.1 Single-turn query

Call the agent with a single question. The agent will decide autonomously whether to invoke the retrieval tool.

In [ ]:
reply = agent("What engineering programmes are available at the Politecnico di Milano?")
print(reply)

At the Politecnico di Milano, there are many engineering programs available, including Aerospace Engineering, Biomedical Engineering, Civil Engineering, Computer Engineering, Electrical Engineering, Environmental Engineering, Industrial Engineering, Mechatronics Engineering, Nuclear Engineering, and more.


---

## 7. Reusability Guidelines

The pattern shown in this tutorial can be adapted to any agent or RAG context. The following points are worth bearing in mind when doing so.

**Function name.** The name should reflect the specific domain of the knowledge base. `search_italian_university_data` is unambiguous; `search_kb` is not. The model reads the name before anything else when deciding which tool to invoke.

**Docstring structure.** Include three components: (1) a one-sentence description of what the tool does; (2) an explicit instruction of *when* to use it (`Use this tool whenever...`); and (3) a description of each parameter and the return format. Without the second component in particular, the model will either over-invoke the tool on irrelevant turns or fail to invoke it when retrieval would genuinely help.

**Return format.** Keep it flat and consistent. A dictionary of numbered string values (`{"doc1": "...", "doc2": "..."}`) is straightforward for the model to reference in its reply. Nested structures or raw lists increase the risk of the model misreading the retrieved content.

**Numeric parameters.** Type any numeric parameter as `str` with internal conversion. This prevents type errors when the model passes values inconsistently across different LLM backends.

**Error handling.** Return an error key rather than raising an exception. This keeps the return type consistent and allows the agent to recover gracefully in the dialogue.